# MBTI Personality Type Classification

## Project Overview

This project implements a machine learning pipeline to classify Myers-Briggs Type Indicator (MBTI) personality types based on text posts from users.

### Objectives
- Download and preprocess the MBTI dataset
- Perform text cleaning and feature extraction using TF-IDF
- Train and evaluate three classic ML models: Logistic Regression, SVM, and Naive Bayes
- Handle class imbalance in the dataset
- Analyze model performance using various metrics

### MBTI Types
The MBTI consists of 16 personality types based on four dimensions:
- **E/I**: Extraversion vs Introversion
- **S/N**: Sensing vs Intuition
- **T/F**: Thinking vs Feeling
- **J/P**: Judging vs Perceiving

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
import os
import joblib
from collections import Counter

import kagglehub
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
from imblearn.over_sampling import RandomOverSampler

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

os.makedirs('images', exist_ok=True)

## 2. Dataset Download & Loading

We'll download the MBTI dataset from Kaggle using `kagglehub` and save it to our data folder.

In [ ]:
path = kagglehub.dataset_download("datasnaek/mbti-type")
csv_file = os.path.join(path, "mbti_1.csv")

df_raw = pd.read_csv(csv_file)

os.makedirs("data", exist_ok=True)
df_raw.to_csv("data/raw_data.csv", index=False)

print(f"Dataset shape: {df_raw.shape}")
df_raw.head()

## 3. Basic Data Exploration

In [ ]:
print(f"Samples: {len(df_raw)} | Features: {len(df_raw.columns)}")
print(f"Columns: {df_raw.columns.tolist()}")
print(f"Missing values: {df_raw.isnull().sum().sum()}")

In [ ]:
type_counts = df_raw['type'].value_counts()
print(type_counts)

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(type_counts)))
bars = ax.bar(type_counts.index, type_counts.values, color=colors)
ax.set_xlabel('MBTI Type')
ax.set_ylabel('Count')
ax.set_title('Distribution of MBTI Personality Types')
plt.xticks(rotation=45)

for bar, count in zip(bars, type_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('images/01_mbti_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Class imbalance ratio (max/min): {type_counts.max() / type_counts.min():.2f}")

In [ ]:
for i in range(3):
    print(f"Type: {df_raw.iloc[i]['type']}")
    print(f"Posts: {df_raw.iloc[i]['posts'][:300]}...\n")

## 4. Data Cleaning & Text Aggregation

We'll perform the following text preprocessing steps:
1. Convert to lowercase
2. Remove URLs
3. Remove punctuation and special characters
4. Remove stopwords
5. Aggregate all posts per user into one document (already done in dataset)

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Clean and preprocess text data."""
    text = text.replace('|||', ' ')
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    mbti_types = ['infp', 'infj', 'intp', 'intj', 'entp', 'enfp', 'istp', 'isfp',
                  'entj', 'istj', 'enfj', 'isfj', 'estp', 'esfp', 'estj', 'esfj']
    for mbti in mbti_types:
        text = re.sub(r'\b' + mbti + r'\b', '', text)
    
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    words = [word for word in words if word not in stop_words and len(word) > 2]
    
    return ' '.join(words)

In [ ]:
df = df_raw.copy()
df["cleaned_posts"] = df["posts"].apply(clean_text)

print(f"Original: {df_raw.iloc[0]['posts'][:200]}...")
print(f"Cleaned: {df.iloc[0]['cleaned_posts'][:200]}...")

df = df[df["cleaned_posts"].str.len() > 0]
print(f"\nDataset size after cleaning: {len(df)}")

In [ ]:
df_processed = df[["type", "cleaned_posts"]].copy()
df_processed.columns = ["type", "posts"]
df_processed.to_csv("data/processed_data.csv", index=False)

print(f"Processed dataset: {df_processed.shape}")
df_processed.head()

## 5. Handling Class Imbalance

The MBTI dataset is imbalanced, with some personality types having significantly more samples than others. We'll use **Random Oversampling** to balance the classes by duplicating samples from minority classes.

In [ ]:
X = df['cleaned_posts'].values
y = df['type'].values

print("Original class distribution:")
print(pd.Series(y).value_counts())

fig, ax = plt.subplots(figsize=(10, 5))
original_counts = pd.Series(y).value_counts().sort_index()
ax.bar(original_counts.index, original_counts.values, color='steelblue')
ax.set_xlabel('MBTI Type')
ax.set_ylabel('Count')
ax.set_title('Original Class Distribution')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('images/02_original_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X.reshape(-1, 1), y)
X_resampled = X_resampled.flatten()

print(f"Original: {len(X)} → Resampled: {len(X_resampled)}")
print(pd.Series(y_resampled).value_counts())

fig, ax = plt.subplots(figsize=(10, 5))
resampled_counts = pd.Series(y_resampled).value_counts().sort_index()
ax.bar(resampled_counts.index, resampled_counts.values, color='forestgreen')
ax.set_xlabel('MBTI Type')
ax.set_ylabel('Count')
ax.set_title('Class Distribution After Random Oversampling')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('images/03_resampled_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. TF-IDF Feature Extraction

We'll convert the text data into numerical features using Term Frequency-Inverse Document Frequency (TF-IDF) vectorization.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8, 
                        ngram_range=(1, 2), sublinear_tf=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF shape - Train: {X_train_tfidf.shape} | Test: {X_test_tfidf.shape}")

## 7. Model Training

We'll train three classic machine learning models:
1. **Logistic Regression** - A simple and interpretable linear model
2. **Support Vector Machine (SVM)** - Effective for high-dimensional text data
3. **Naive Bayes** - Fast and works well with text classification

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'SVM': LinearSVC(max_iter=2000, random_state=42),
    'Naive Bayes': MultinomialNB()
}

results = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    
    results[name] = {
        'model': model,
        'predictions': y_pred,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
        'f1': f1_score(y_test, y_pred, average='weighted')
    }
    print(f"{name}: Accuracy = {results[name]['accuracy']:.4f}")

## 8. Model Evaluation

We'll evaluate each model using multiple metrics: Accuracy, Precision, Recall, F1-score, and Confusion Matrix.

In [ ]:
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results]
}).round(4).sort_values('F1-Score', ascending=False)

print(comparison_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25

for i, (name, result) in enumerate(results.items()):
    values = [result['accuracy'], result['precision'], result['recall'], result['f1']]
    bars = ax.bar(x + i * width, values, width, label=name)
    for j, v in enumerate(values):
        ax.text(j + i * width, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('images/04_model_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for name in results:
    print(f"\n{'='*50}\n{name.upper()}\n{'='*50}")
    print(classification_report(y_test, results[name]['predictions']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
mbti_types = sorted(df['type'].unique())

for ax, (name, result) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, result['predictions'], labels=mbti_types)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=mbti_types, yticklabels=mbti_types, annot_kws={'size': 7})
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{name} (Acc: {result["accuracy"]:.4f})')
    ax.tick_params(axis='both', labelsize=8)
plt.tight_layout()
plt.savefig('images/05_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Error Analysis

Let's analyze the types of errors our best model makes to understand its limitations.

In [ ]:
best_model_name = max(results, key=lambda x: results[x]['f1'])
best_result = results[best_model_name]
y_pred_best = best_result['predictions']

misclassified_idx = np.where(y_test != y_pred_best)[0]
print(f"Best Model: {best_model_name} (F1: {best_result['f1']:.4f})")
print(f"Misclassified: {len(misclassified_idx)}/{len(y_test)} ({len(misclassified_idx)/len(y_test)*100:.2f}%)")

In [ ]:
misclassifications = [(y_test[i], y_pred_best[i]) for i in misclassified_idx]
misclass_counts = Counter(misclassifications)

print("Top 10 Misclassifications:")
for (true_type, pred_type), count in misclass_counts.most_common(10):
    print(f"  {true_type} → {pred_type}: {count}")

In [ ]:
type_accuracy = {}
for mbti in mbti_types:
    mask = y_test == mbti
    if mask.sum() > 0:
        type_accuracy[mbti] = (y_test[mask] == y_pred_best[mask]).mean()

type_accuracy_sorted = dict(sorted(type_accuracy.items(), key=lambda x: x[1]))

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['crimson' if v < 0.3 else 'orange' if v < 0.5 else 'forestgreen' for v in type_accuracy_sorted.values()]
bars = ax.barh(list(type_accuracy_sorted.keys()), list(type_accuracy_sorted.values()), color=colors)
ax.set_xlabel('Accuracy')
ax.set_title(f'Per-Type Classification Accuracy ({best_model_name})')
ax.set_xlim(0, 1)

for bar, acc in zip(bars, type_accuracy_sorted.values()):
    ax.text(acc + 0.01, bar.get_y() + bar.get_height()/2, f'{acc:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('images/06_per_type_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Best: {max(type_accuracy, key=type_accuracy.get)} ({max(type_accuracy.values()):.4f})")
print(f"Worst: {min(type_accuracy, key=type_accuracy.get)} ({min(type_accuracy.values()):.4f})")

In [ ]:
def get_dimension(mbti_type, dim_idx):
    return mbti_type[dim_idx]

dimensions = ['E/I', 'S/N', 'T/F', 'J/P']
dim_accuracy = {}

for i, dim_name in enumerate(dimensions):
    true_dim = np.array([get_dimension(t, i) for t in y_test])
    pred_dim = np.array([get_dimension(t, i) for t in y_pred_best])
    dim_accuracy[dim_name] = (true_dim == pred_dim).mean()

print("Dimension-Level Accuracy:")
for dim, acc in dim_accuracy.items():
    print(f"  {dim}: {acc:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.Set2(np.linspace(0, 1, 4))
bars = ax.bar(dim_accuracy.keys(), dim_accuracy.values(), color=colors)
ax.set_ylabel('Accuracy')
ax.set_title(f'Dimension-Level Accuracy ({best_model_name})')
ax.set_ylim(0, 1)

for bar, acc in zip(bars, dim_accuracy.values()):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f'{acc:.3f}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('images/07_dimension_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Conclusion

### Summary of Results

In [ ]:
print("=" * 50)
print("FINAL SUMMARY")
print("=" * 50)

print(f"\n📊 Dataset: {len(df_raw)} → {len(df)} (cleaned) → {len(X_resampled)} (resampled)")
print(f"📦 Train/Test: {len(X_train)} / {len(X_test)}")
print(f"🔧 TF-IDF Features: {X_train_tfidf.shape[1]}")

print("\n🤖 Model Results:")
for name in comparison_df['Model']:
    row = comparison_df[comparison_df['Model'] == name].iloc[0]
    print(f"   {name}: Acc={row['Accuracy']:.4f}, F1={row['F1-Score']:.4f}")

print(f"\n🏆 Best: {best_model_name}")
print(f"   Accuracy: {best_result['accuracy']:.4f}")
print(f"   F1-Score: {best_result['f1']:.4f}")

print(f"\n📈 Dimension Accuracy: {max(dim_accuracy, key=dim_accuracy.get)} best ({max(dim_accuracy.values()):.2%})")
print("=" * 50)

### Key Findings

1. **Model Performance**: All three classic ML models (Logistic Regression, SVM, and Naive Bayes) were trained and evaluated on the MBTI dataset.

2. **Class Imbalance**: The original dataset showed significant class imbalance, which was addressed using Random Oversampling.

3. **Feature Extraction**: TF-IDF vectorization with unigrams and bigrams was used to convert text into numerical features.

4. **Challenges**: MBTI classification from text is inherently difficult because:
   - Personality traits are nuanced and may not always be clearly expressed in text
   - The four MBTI dimensions have varying degrees of detectability from writing style
   - Some dimensions (like E/I or T/F) may be easier to detect than others

### Limitations

- Self-reported MBTI types may not be accurate
- Online forum posts may not represent natural writing style
- The dataset is limited to a specific online community

### Future Improvements

- Try different text preprocessing techniques
- Use word embeddings instead of TF-IDF
- Focus on binary classification for individual dimensions
- Collect more diverse data sources

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_result['model'], f'models/{best_model_name.lower().replace(" ", "_")}_model.pkl')
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')
print("✅ Model and vectorizer saved to models/")